# Content-Yield → Full-Range Pack (headless GPU) — v5

Builds the **authoritative full-range flop content pack** on a Kaggle **T4 GPU**, with **per-board checkpointing** so a crash or the session time-limit never costs you more than the board in flight.

**Run (headless — survives a closed browser):**
1. Settings → **Accelerator → GPU T4 x1**, **Internet → On** (one-time phone verification may be required).
2. **Save Version → Save & Run All (Commit)**. Close the tab; it runs in the background.
3. When done: version → **Output** tab → download `flop_pack_v1_fullrange.db` (+ `.db.gz`, `records_v1_fullrange.json`).

**Robustness built in:** each board is solved, validated (NaN/inf records dropped, never leaked into the signed pack), then written to `cyout/boards/board_XX.json`. `records.json` + `yield_report.json` are refreshed **after every board**, so even a mid-run timeout leaves valid, downloadable partial output. If one board crashes it's logged (`board_XX.ERROR.txt`) and the run **continues** with the rest. Config: full ranges, all 12 boards, check/bet/fold/call, 300 iters, float32 (~5–7 h — well inside Kaggle's 12 h limit).

In [ ]:
!nvidia-smi -L || echo 'NO GPU — Settings -> Accelerator -> GPU T4 x1'

In [ ]:
# Unpack the current solver source (all fixes) into /kaggle/working/poker
import base64, zipfile, io, os
_ZIP_B64 = (
    "..."
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_ZIP_B64))) as z: z.extractall('/kaggle/working/poker')
print('unpacked ->', sorted(os.listdir('/kaggle/working/poker/src/pokertrainer'))[:8], '...')


In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

### Smoke (optional, ~20–35 min) — 1 board at full range, float32. Confirms the GPU path before the long run.

In [ ]:
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float32 --n 400 --iters 300 --roots 0 --out /kaggle/working/cy_smoke
import cupy as cp; print('peak GPU mem used: %.1f GB'%(cp.get_default_memory_pool().used_bytes()/1e9))
import json; r=json.load(open('/kaggle/working/cy_smoke/yield_report.json'))
print(r['config'], '\n', r['projected_raw_accepted_per_root_full_range'],'raw accepted/root (full range)')

## Full run — 12 boards, full range, float32 (checkpointed, ~5–7 h)

In [ ]:
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float32 --n 400 --iters 300 --out /kaggle/working/cyout

In [ ]:
# Build + sign + verify the pack from whatever boards completed (partial-safe), expose for download.
import subprocess, shutil, os, json, sys
env={**os.environ,'PYTHONPATH':'src'}
rep=json.load(open('/kaggle/working/cyout/yield_report.json'))
print('boards completed:', rep.get('boards_completed'), '| missing:', rep.get('boards_missing'))
subprocess.run(['python','-m','pokertrainer.content_pack',
                '--records','/kaggle/working/cyout/records.json',
                '--version','v1_fullrange','--out','/kaggle/working'],
               cwd='/kaggle/working/poker', env=env, check=True)
shutil.copy('/kaggle/working/cyout/records.json','/kaggle/working/records_v1_fullrange.json')
sys.path.insert(0,'/kaggle/working/poker/src')
from pokertrainer.content_pack import verify_pack
db='/kaggle/working/flop_pack_v1_fullrange.db'
print('VERIFY:', verify_pack(db), '| size MB:', round(os.path.getsize(db)/1e6,2))
print(json.dumps({k:rep[k] for k in ('accepted','accepted_deduped','distinct_concepts_measured','per_node_accepted','coverage')}, indent=1))

---
### Optional — raise-inclusive full range (FR-011)
fold/call/**raise** ~triples solve time, so all 12 boards won't fit one 12 h commit. Run this in **two commits**: `HALF='A'` (boards 0–5), commit; then `HALF='B'` (boards 6–11), commit. Checkpointing means each commit resumes any boards it already finished. Download both `records_raise_*.json`; the raise pack is merged + signed locally afterwards.

In [ ]:
HALF='A'   # <-- 'A' for one commit, 'B' for the other
ROOTS={'A':'0,1,2,3,4,5','B':'6,7,8,9,10,11'}[HALF]
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float32 --n 400 --iters 300 --raise-x 3 --roots $ROOTS \
    --out /kaggle/working/cy_raise_$HALF
import shutil; shutil.copy(f'/kaggle/working/cy_raise_{HALF}/records.json', f'/kaggle/working/records_raise_{HALF}.json')
print('done half', HALF)